In [25]:
import pandas as pd
import pickle
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import NearestNeighbors

In [26]:
df=pd.read_csv(r"C:\Users\HP\Downloads\movies_data.csv")

In [27]:
print(df)

                             Movie Name Release Period Whether Remake  \
0                           Golden Boys         Normal             No   
1                         Kaccha Limboo        Holiday             No   
2                      Not A Love Story        Holiday             No   
3                            Qaidi Band        Holiday             No   
4                             Chaatwali        Holiday             No   
...                                 ...            ...            ...   
1693                         Fight Club        Holiday             No   
1694                 Strings Of Paasion         Normal             No   
1695              Dunno Y Na Jaane Kyun         Normal             No   
1696  Taj Mahal - An Eternal Love Story         Normal             No   
1697                   Mr. Hot Mr. Kool         Normal             No   

     Whether Franchise     Genre New Actor New Director New Music Director  \
0                   No  suspense       Yes   

In [28]:
##Data cleaning
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Movie Name          1698 non-null   object
 1   Release Period      1698 non-null   object
 2   Whether Remake      1698 non-null   object
 3   Whether Franchise   1698 non-null   object
 4   Genre               1698 non-null   object
 5   New Actor           1698 non-null   object
 6   New Director        1698 non-null   object
 7   New Music Director  1698 non-null   object
 8   Lead Star           1698 non-null   object
 9   Director            1698 non-null   object
 10  Music Director      1698 non-null   object
 11  Number of Screens   1698 non-null   int64 
 12  Revenue(INR)        1698 non-null   int64 
 13  Budget(INR)         1698 non-null   int64 
dtypes: int64(3), object(11)
memory usage: 185.8+ KB


In [29]:
df.describe()

,Number of Screens,Revenue(INR),Budget(INR)
count,1698.000000,1.698000e+03,1.698000e+03
mean,553.831567,1.501674e+08,2.377287e+08
std,782.951839,2.434838e+08,6.134398e+08
min,1.000000,3.250000e+05,7.250000e+03
25%,30.000000,1.500000e+07,1.150000e+06
50%,200.000000,5.500000e+07,1.240000e+07
75%,800.000000,1.900000e+08,1.778325e+08
max,4600.000000,2.100000e+09,8.016120e+09


In [30]:
df.isnull().sum()

Movie Name            0
Release Period        0
Whether Remake        0
Whether Franchise     0
Genre                 0
New Actor             0
New Director          0
New Music Director    0
Lead Star             0
Director              0
Music Director        0
Number of Screens     0
Revenue(INR)          0
Budget(INR)           0
dtype: int64

In [31]:
df.duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
1693    False
1694    False
1695    False
1696    False
1697    False
Length: 1698, dtype: bool

In [32]:
categorical_columns = [
    "Release Period",
    "Whether Remake",
    "Whether Franchise",
    "Genre",
    "New Actor",
    "New Director",
    "New Music Director"
]

df_encoded = pd.get_dummies(df, columns=categorical_columns)

In [33]:
X = df_encoded.drop(columns=[
    "Movie Name",
    "Lead Star",
    "Director",
    "Music Director"
])
print(X)

      Number of Screens  Revenue(INR)  Budget(INR)  Release Period_Holiday  \
0                     5       5000000        85000                   False   
1                    75      15000000       825000                    True   
2                   525      75000000     56700000                    True   
3                   800     210000000      4500000                    True   
4                     1       1000000      1075000                    True   
...                 ...           ...          ...                     ...   
1693                375      82500000     88862500                    True   
1694                 10       8000000        70000                   False   
1695                 20      12500000       850000                   False   
1696                135     100000000     31065000                   False   
1697                 30      27500000      1300000                   False   

      Release Period_Normal  Whether Remake_No  Whether Remake_

In [34]:
from sklearn.neighbors import NearestNeighbors

knn = NearestNeighbors(n_neighbors=5)

knn.fit(X)

NearestNeighbors()

In [35]:
pickle.dump(knn, open("movie_knn.pkl", "wb"))
pickle.dump(df, open("movie_data.pkl", "wb"))
pickle.dump(X.columns.tolist(), open("encoded_columns.pkl", "wb"))

print("Model Saved Successfully!")

Model Saved Successfully!


In [36]:
from tkinter import *
from tkinter import ttk
import pandas as pd
import pickle

# ===================== LOAD FILES =====================

knn = pickle.load(open("movie_knn.pkl", "rb"))
movies = pickle.load(open("movie_data.pkl", "rb"))
encoded_columns = pickle.load(open("encoded_columns.pkl", "rb"))

# ===================== FUNCTION =====================

def recommend():

    result.delete(1.0, END)

    genre = genre_box.get()

    if genre == "":
        result.insert(END, "Please Select a Genre")
        return

    # Create input vector
    sample = pd.DataFrame(0, index=[0], columns=encoded_columns)

    genre_column = "Genre_" + genre

    if genre_column in sample.columns:
        sample[genre_column] = 1

    # Find nearest movies
    distance, index = knn.kneighbors(sample)

    result.insert(END, "Recommended Movies\n")
    result.insert(END, "=" * 70 + "\n\n")

    for i in index[0]:

        movie = movies.iloc[i]

        result.insert(
            END,
            f"""
🎬 Movie Name      : {movie['Movie Name']}

🎭 Genre           : {movie['Genre']}

⭐ Lead Star       : {movie['Lead Star']}

🎥 Director        : {movie['Director']}

🎼 Music Director  : {movie['Music Director']}

💰 Budget          : ₹ {movie['Budget(INR)']}

📈 Revenue         : ₹ {movie['Revenue(INR)']}

------------------------------------------------------------------

"""
        )


# ===================== GUI =====================

root = Tk()

root.title("Movie Recommendation System")

root.geometry("1000x700")

root.configure(bg="#0B132B")

# ===================== Heading =====================

heading = Label(
    root,
    text="🎬 MOVIE RECOMMENDATION SYSTEM",
    font=("Georgia", 26, "bold"),
    bg="#0B132B",
    fg="white"
)

heading.pack(pady=15)

sub = Label(
    root,
    text="Discover Movies Based on Your Favourite Genre",
    font=("Arial", 13),
    bg="#0B132B",
    fg="lightgray"
)

sub.pack()

# ===================== Search Frame =====================

search_frame = Frame(
    root,
    bg="#1C2541",
    bd=3,
    relief=RIDGE
)

search_frame.pack(fill="x", padx=20, pady=20)

Label(
    search_frame,
    text="Select Genre",
    font=("Arial",16,"bold"),
    bg="#1C2541",
    fg="white"
).grid(row=0,column=0,padx=20,pady=20)

genres = sorted(movies["Genre"].unique())

genre_box = ttk.Combobox(
    search_frame,
    values=genres,
    width=28,
    font=("Arial",13),
    state="readonly"
)

genre_box.grid(row=0,column=1,padx=20)

Button(
    search_frame,
    text="🔍 Recommend Movies",
    bg="#FF6B00",
    fg="white",
    font=("Arial",13,"bold"),
    padx=20,
    pady=5,
    command=recommend
).grid(row=0,column=2,padx=20)

# ===================== Result Frame =====================

result_frame = LabelFrame(
    root,
    text=" Recommended Movies ",
    font=("Arial",14,"bold"),
    bg="#0B132B",
    fg="white"
)

result_frame.pack(fill="both", expand=True, padx=20, pady=10)

scroll = Scrollbar(result_frame)

scroll.pack(side=RIGHT, fill=Y)

result = Text(
    result_frame,
    width=95,
    height=22,
    font=("Consolas",12),
    yscrollcommand=scroll.set
)

result.pack(fill=BOTH, expand=True)

scroll.config(command=result.yview)

# ===================== Footer =====================

footer = Label(
    root,
    text="Developed using K-Nearest Neighbors (KNN)",
    font=("Arial",11),
    bg="#0B132B",
    fg="white"
)

footer.pack(pady=10)

root.mainloop()